# lyp-sync — Lip Sync from Image + Audio

Generate a talking-head video from a **portrait image** and **speech audio** (MP3 or WAV).

Powered by [SadTalker](https://github.com/OpenTalker/SadTalker) on Google Colab.

---

### Before you start
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells **1 → 6 in order** (do not skip)
3. A ~60s clip takes about **10–30 min** on a free T4 GPU

In [ ]:
# @title 1. Check GPU
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU ready: {name} ({vram:.1f} GB)")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

In [ ]:
# @title 2. Install SadTalker (~5 min first run)
import importlib
import os
import subprocess
import sys
from pathlib import Path

# --- paths (used by later cells) ---
ROOT = Path("/content/lyp-sync")
SADTALKER = ROOT / "SadTalker"
CHECKPOINTS = ROOT / "checkpoints"
UPLOADS = ROOT / "uploads"
OUTPUTS = ROOT / "outputs"

for d in (ROOT, CHECKPOINTS, UPLOADS, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

def run(cmd, **kwargs):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd, **kwargs)

if not (SADTALKER / "inference.py").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/OpenTalker/SadTalker.git", str(SADTALKER)])
else:
    print("SadTalker already cloned.")

run(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# Do NOT reinstall torch — Colab ships a CUDA build.
# SadTalker's pinned requirements.txt breaks on modern Colab.
pip = [sys.executable, "-m", "pip", "install", "-q"]

run(pip + ["setuptools", "wheel", "cython"])
run(pip + [
    "numpy", "scipy", "tqdm", "yacs", "pyyaml", "joblib", "pydub",
    "numba", "resampy", "librosa", "scikit-image", "opencv-python",
    "kornia", "facexlib", "safetensors", "av", "face_alignment",
    "imageio>=2.31", "imageio-ffmpeg>=0.4.9", "ipywidgets",
])

# gfpgan + basicsr: install separately with build workaround
try:
    run(pip + ["gfpgan", "basicsr", "--no-build-isolation"])
except subprocess.CalledProcessError:
    print("Retrying basicsr from GitHub...")
    run(pip + ["git+https://github.com/XPixelGroup/BasicSR.git@v1.4.2", "--no-build-isolation"])
    run(pip + ["gfpgan"])

# Patch basicsr for newer torchvision
import basicsr

degradations = Path(basicsr.__file__).parent / "data/degradations.py"
text = degradations.read_text()
old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
new = """try:
    from torchvision.transforms.functional_tensor import rgb_to_grayscale
except ImportError:
    from torchvision.transforms.functional import rgb_to_grayscale"""
if old in text:
    degradations.write_text(text.replace(old, new, 1))
    print("Patched basicsr.")

# Verify critical imports
for mod in ["torch", "basicsr", "face_alignment", "librosa", "kornia", "safetensors", "imageio"]:
    importlib.import_module(mod)
    print(f"  ok: {mod}")

print("\nInstall complete.")

In [ ]:
# @title 3. Download model checkpoints (~1.5 GB)
import subprocess
from pathlib import Path

# Re-define paths in case this cell is re-run alone
ROOT = Path("/content/lyp-sync")
CHECKPOINTS = ROOT / "checkpoints"
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

RELEASE = "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc"
files = {
    "SadTalker_V0.0.2_256.safetensors": 700_000_000,
    "SadTalker_V0.0.2_512.safetensors": 700_000_000,
    "mapping_00109-model.pth.tar": 120_000_000,
    "mapping_00229-model.pth.tar": 120_000_000,
}

for name, min_bytes in files.items():
    dest = CHECKPOINTS / name
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f"skip: {name} ({dest.stat().st_size // 1_000_000} MB)")
        continue
    print(f"download: {name}")
    subprocess.check_call([
        "wget", "--show-progress", "-O", str(dest), f"{RELEASE}/{name}"
    ])
    if dest.stat().st_size < min_bytes:
        dest.unlink(missing_ok=True)
        raise RuntimeError(f"Download incomplete for {name}. Re-run this cell.")

print("Checkpoints ready.")

In [ ]:
# @title 4. Upload image + audio
from pathlib import Path
from google.colab import files

UPLOADS = Path("/content/lyp-sync/uploads")
UPLOADS.mkdir(parents=True, exist_ok=True)

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg"}

def _save_upload(label, allowed_ext):
    print(label)
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    name = next(iter(uploaded))
    ext = Path(name).suffix.lower()
    if ext not in allowed_ext:
        raise ValueError(f"Unsupported file '{name}'. Allowed: {sorted(allowed_ext)}")
    dest = UPLOADS / name
    dest.write_bytes(uploaded[name])
    print(f"  → {dest}")
    return dest

image_path = _save_upload("Upload portrait image (JPG/PNG)...", IMAGE_EXT)
audio_path = _save_upload("\nUpload speech audio (MP3/WAV)...", AUDIO_EXT)

In [ ]:
# @title 5. Settings
size = 256  # @param [256, 512] {type:"raw"}
preprocess = "crop"  # @param ["crop", "resize", "full", "extcrop", "extfull"]
still_mode = False  # @param {type:"boolean"}
use_enhancer = False  # @param {type:"boolean"}
batch_size = 2  # @param {type:"slider", min:1, max:4, step:1}
output_name = "lipsync_output.mp4"  # @param {type:"string"}

In [ ]:
# @title 6. Generate lip-sync video
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import files
from IPython.display import Video, display

# Re-define paths
ROOT = Path("/content/lyp-sync")
SADTALKER = ROOT / "SadTalker"
CHECKPOINTS = ROOT / "checkpoints"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

for var in ("image_path", "audio_path", "size", "preprocess", "batch_size", "output_name"):
    if var not in globals():
        raise RuntimeError(f"Missing '{var}'. Run cells 4 and 5 first.")

if not (SADTALKER / "inference.py").exists():
    raise RuntimeError("SadTalker not installed. Run cell 2 first.")

cmd = [
    sys.executable, "-u", str(SADTALKER / "inference.py"),
    "--driven_audio", str(audio_path),
    "--source_image", str(image_path),
    "--checkpoint_dir", str(CHECKPOINTS),
    "--result_dir", str(OUTPUTS),
    "--size", str(size),
    "--preprocess", preprocess,
    "--batch_size", str(batch_size),
]
if still_mode:
    cmd.append("--still")
if use_enhancer:
    cmd.extend(["--enhancer", "gfpgan"])

print("Running SadTalker...")
print(" ".join(cmd))
print("-" * 60)

before = set(OUTPUTS.glob("*.mp4"))
env = os.environ.copy()
env["PYTHONPATH"] = str(SADTALKER)
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.run(cmd, cwd=str(SADTALKER), env=env)
if proc.returncode != 0:
    raise RuntimeError(f"SadTalker failed with exit code {proc.returncode}")

new_files = set(OUTPUTS.glob("*.mp4")) - before
results = sorted(new_files or OUTPUTS.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not results:
    raise RuntimeError("No output video found in outputs/")

final = OUTPUTS / output_name
shutil.copy2(results[-1], final)
print(f"\nSaved: {final}")
display(Video(str(final), embed=True, width=512))

print("\nDownload:")
files.download(str(final))